# Parallel Topic Analyzer - Interactive Demo

This notebook demonstrates the parallel topic analyzer agent using LangGraph and Ollama.

## Overview

The agent executes three LLM tasks **in parallel**:
1. Summarize the topic
2. Generate questions about the topic
3. Extract key terms

All results are combined into structured JSON output.

## Architecture

```
START → [summarize, questions, key_terms] → combine → END
```

**Parallel execution**: ~3x speedup vs sequential!

## Setup

In [ ]:
import sys
import json
from pathlib import Path

# Add parent directory to path for imports
parent_dir = Path().resolve().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

# Import our agent
from src.agent import TopicAnalyzerAgent

print("✓ Imports successful")

## Check Ollama Connection

In [ ]:
import requests

OLLAMA_URL = "http://localhost:11434"

# Check server
try:
    response = requests.get(f"{OLLAMA_URL}/api/version", timeout=5)
    print(f"✓ Ollama server running: v{response.json()['version']}")
except Exception as e:
    print(f"✗ Ollama server not reachable: {e}")
    print("  Please run: ollama serve")

# List available models
try:
    response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    models = response.json().get("models", [])
    print(f"\n✓ Available models ({len(models)}):")
    for model in models[:5]:  # Show first 5
        name = model["name"]
        size_gb = model["size"] / (1024**3)
        print(f"  - {name} ({size_gb:.1f}GB)")
except Exception as e:
    print(f"✗ Could not list models: {e}")

## Initialize Agent

In [ ]:
# Create agent with default model
agent = TopicAnalyzerAgent(
    ollama_base_url="http://localhost:11434",
    model="deepseek-r1:1.5b"  # Change to deepseek-r1:8b for better quality
)

print(f"✓ Agent initialized with model: {agent.model}")
print(f"✓ LangGraph compiled with {len(agent.graph.nodes)} nodes")

## Example 1: Analyze "Artificial Intelligence"

In [ ]:
import time

topic = "Artificial Intelligence"
print(f"Analyzing: {topic}")
print("Running 3 parallel LLM tasks...\n")

start = time.time()
result = agent.analyze(topic)
elapsed = time.time() - start

print(f"✓ Completed in {elapsed:.2f}s")
print("\nResults:")
print(json.dumps(result, indent=2, ensure_ascii=False))

## Example 2: Analyze "Quantum Computing"

In [ ]:
topic = "Quantum Computing"
result = agent.analyze(topic)

print(f"Topic: {result['topic']}")
print(f"\nSummary:\n{result['results']['summary']}")
print(f"\nQuestions:")
for i, q in enumerate(result['results']['questions'], 1):
    print(f"  {i}. {q}")
print(f"\nKey Terms: {', '.join(result['results']['key_terms'])}")
print(f"\nExecution Time: {result['metadata']['execution_time_seconds']}s")

## Example 3: Compare Multiple Topics

In [ ]:
topics = [
    "Climate Change",
    "Machine Learning",
    "Renewable Energy"
]

results = []
for topic in topics:
    print(f"Analyzing: {topic}...")
    result = agent.analyze(topic)
    results.append(result)
    print(f"  ✓ Done in {result['metadata']['execution_time_seconds']}s")

print("\n" + "="*60)
print("COMPARISON")
print("="*60)

for result in results:
    print(f"\n{result['topic']}:")
    print(f"  Key Terms: {', '.join(result['results']['key_terms'][:3])}...")
    print(f"  Time: {result['metadata']['execution_time_seconds']}s")

## Inspect the LangGraph Structure

In [ ]:
# Show graph structure
print("Graph Nodes:")
for node in agent.graph.nodes:
    print(f"  - {node}")

print("\nGraph Edges:")
for edge in agent.graph.edges:
    print(f"  - {edge}")

# Note: To visualize the graph as an image, you would need:
# agent.graph.get_graph().draw_mermaid_png()
# This requires graphviz and additional setup

## Custom Configuration

In [ ]:
# Try with a different model (if you have it)
agent_8b = TopicAnalyzerAgent(
    ollama_base_url="http://localhost:11434",
    model="deepseek-r1:8b"  # Larger model = better quality
)

result = agent_8b.analyze("Neural Networks")
print(f"Model: {result['metadata'].get('model', agent_8b.model)}")
print(f"Summary: {result['results']['summary'][:200]}...")

## Performance Analysis

In [ ]:
# Run same topic multiple times to see consistency
import pandas as pd

topic = "Blockchain Technology"
runs = []

print(f"Running 3 trials for: {topic}\n")
for i in range(3):
    print(f"Trial {i+1}...", end=" ")
    result = agent.analyze(topic)
    runs.append({
        "trial": i+1,
        "time": result['metadata']['execution_time_seconds'],
        "questions_count": len(result['results']['questions']),
        "terms_count": len(result['results']['key_terms'])
    })
    print(f"✓ {runs[-1]['time']}s")

# Show statistics
df = pd.DataFrame(runs)
print("\nPerformance Statistics:")
print(df)
print(f"\nAverage time: {df['time'].mean():.2f}s")
print(f"Std deviation: {df['time'].std():.2f}s")

## Next Steps

Try experimenting with:

1. **Different topics**: Scientific, historical, technical, cultural
2. **Different models**: Compare `deepseek-r1:1.5b` vs `deepseek-r1:8b` vs `granite3.2:2b`
3. **Model parameters**: Adjust temperature in `src/tasks.py`
4. **Add more tasks**: Modify the graph to include additional parallel nodes
5. **Streaming**: Implement real-time streaming of results

## Learn More

- **Example README**: `../README.md`
- **Usage Examples**: `../EXAMPLES.md`
- **Ollama Internals**: `../OLLAMA_INTERNALS.md`
- **LangGraph Docs**: https://langchain-ai.github.io/langgraph/